# Exercicis: diagnòstic de càncer de mama

**Optativa d'Aprenentatge automàtic — DAM/DAW 2n**

Fins ara heu mesurat els models amb un sol número: la precisió (percentatge
d'encerts). Aquest quadern fa servir un dataset mèdic real per ensenyar per
què, en molts problemes, **la precisió sola no serveix de res**.

El conjunt és `load_breast_cancer` de scikit-learn: 569 tumors de mama,
descrits per 30 mesures fetes sobre la imatge de les seves cèl·lules (radi,
textura, perímetre, concavitat...), classificats com a **malignes** o
**benignes**. És un problema de classificació binària, com "spam o no spam",
però aquí un error no val el mateix que l'altre: **deixar passar un tumor
maligne (fals negatiu) és molt més greu que donar una falsa alarma sobre un
de benigne (fals positiu)**.

Fes els exercicis en ordre: el 6 no té sentit sense el 5, i el 5 sense el 4.
Tot funciona a Google Colab sense instal·lar res.

### Exercici 1 — Carregar i explorar les dades

Carrega el dataset amb `load_breast_cancer(as_frame=True)`. `data.frame` ja
et dona un DataFrame amb les 30 columnes de mesures més una columna
`target`. Guarda'l a `dades` i mira quantes files i columnes té.

Després, compta quantes mostres hi ha de cada valor de `target`
(`dades["target"].value_counts()`). Fixa-t'hi: **el dataset no està
equilibrat**.

Finalment, el parany: `target` és un número (0 o 1), no diu per si sol si
és maligne o benigne. Mira `data.target_names` per esbrinar quin número és
quin diagnòstic, i escriu-ho en una frase.

In [ ]:
# Exercici 1
from sklearn.datasets import load_breast_cancer

data = load_breast_cancer(as_frame=True)

# 'dades' amb totes les columnes + target (data.frame ja el porta fet)
dades = ...

# quantes files i columnes té 'dades'?


# quantes mostres hi ha de cada valor de target?


# quins noms té cada valor de target? (data.target_names)

**Quin número és maligne i quin és benigne?**

_Escriu aquí la teva resposta, mirant `data.target_names`._

**Com saps que ho has fet bé:** 569 mostres i 31 columnes a `dades` (30 de
mesures més `target`). El repartiment és 357 benignes i 212 malignes: hi ha
un 62,7 % de benignes, més d'una i mitja vegada les malignes.

### Exercici 2 — Primer model, i la seva precisió

Separa les dades en entrenament i examen (`test_size=0.3, random_state=42`),
fent servir `data.data` com a `X` i `data.target` com a `y`. Entrena un
`DecisionTreeClassifier(random_state=42)` i mesura la seva precisió sobre
l'examen amb `.score()`.

In [ ]:
# Exercici 2
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

X = data.data
y = data.target

X_entrena, X_examen, y_entrena, y_examen = ...

model_arbre = DecisionTreeClassifier(random_state=42)
# entrena'l i calcula la precisió sobre l'examen

precisio_arbre = ...
print(f"Precisió de l'arbre: {precisio_arbre:.1%}")

**Com saps que ho has fet bé:** una precisió del **94,2 %**. Sembla un molt
bon model. Al següent exercici mirem **quins** errors comet exactament,
perquè no tots pesen igual.

### Exercici 3 — La matriu de confusió

La precisió amaga els detalls de cada error. La **matriu de confusió**
els mostra tots: files = diagnòstic real, columnes = diagnòstic predit, en
l'ordre `[maligne, benigne]` (recorda l'exercici 1: `0 = maligne`).

Calcula-la amb `confusion_matrix(y_examen, prediccions, labels=[0, 1])` del
model de l'exercici 2, i també imprimeix `classification_report` (dona la
mateixa informació en forma de taula). A la matriu, la casella
`[fila=maligne, columna=benigne]` són els **falsos negatius**: tumors que
eren maligne i el model ha dit que eren benignes. És el pitjor error
possible aquí: el pacient se'n va a casa sense tractament.

**Quants falsos negatius té aquest model?**

In [ ]:
# Exercici 3
from sklearn.metrics import confusion_matrix, classification_report

prediccions_arbre = ...

matriu = confusion_matrix(y_examen, prediccions_arbre, labels=[0, 1])
print(matriu)

print(classification_report(y_examen, prediccions_arbre,
                             target_names=data.target_names))

falsos_negatius_arbre = ...  # agafa'l directament de la matriu
print(f"Falsos negatius: {falsos_negatius_arbre}")

**Com saps que ho has fet bé:** la matriu surt `[[60, 3], [7, 101]]`: **3
falsos negatius** (tumors maligne que ha dit benignes) i 7 falsos positius.
El `classification_report` diu el mateix amb un altre format: el
`recall` de `malignant` és `0.95`, és a dir, 60 de 63 encerts sobre els
malignes reals — els mateixos 3 que falten.

### Exercici 4 — El model ximple

Entrena un `DummyClassifier(strategy="most_frequent")`: un "model" que ni
mira les dades, sempre respon la classe més freqüent de l'entrenament.
Mesura'n la precisió sobre l'examen igual que a l'exercici 2.

**Serveix de res aquest model?** Compara la seva precisió amb la de
l'arbre de l'exercici 2, i pensa quants falsos negatius deu tenir un model
que sempre diu "benigne".

In [ ]:
# Exercici 4
from sklearn.dummy import DummyClassifier

model_ximple = DummyClassifier(strategy="most_frequent")
# entrena'l i calcula la precisió sobre l'examen

precisio_ximple = ...
print(f"Precisió del model ximple: {precisio_ximple:.1%}")

**Serveix de res aquest model?**

_Escriu aquí la teva resposta (dues o tres línies)._

**Com saps que ho has fet bé:** precisió del **63,2 %** — que coincideix
amb el percentatge de benignes que hi ha al conjunt d'examen (108 de 171).
Si en calcules la matriu de confusió (no cal fer-ho per a l'exercici, però
prova-ho), veuràs 63 falsos negatius sobre 63 malignes — cap encert entre
els malignes.

### Exercici 5 — Comparar els cinc models per sensibilitat, no per precisió

Entrena els cinc models de la teoria (k-NN, arbre de decisió, bosc
aleatori, regressió logística i SVM) sobre aquestes mateixes dades. Escala
les dades amb `StandardScaler` per a k-NN, la regressió logística i l'SVM
(l'arbre i el bosc no ho necessiten).

Per a cada model calcula tres números sobre l'examen:
- la **precisió** (`.score()` o `accuracy_score`),
- la **sensibilitat sobre la classe maligne**, també dita *recall*: de tots
  els tumors realment malignes, quants n'ha trobat
  (`recall_score(y_examen, prediccions, pos_label=0)`),
- la **precisió del positiu** sobre la classe maligne: dels que ha dit
  malignes, quants ho eren de veritat
  (`precision_score(y_examen, prediccions, pos_label=0)`).

Munta una taula (un DataFrame amb una fila per model) i ordena-la primer
per precisió i després per sensibilitat. **Canvia el rànquing?**

In [ ]:
# Exercici 5
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, recall_score, precision_score

escalador = StandardScaler()
X_entrena_esc = escalador.fit_transform(X_entrena)
X_examen_esc = escalador.transform(X_examen)

# (model, necessita_escalar)
models = {
    "kNN": (KNeighborsClassifier(n_neighbors=5), True),
    "Arbre de decisió": (DecisionTreeClassifier(random_state=42), False),
    "Bosc aleatori": (RandomForestClassifier(random_state=42), False),
    "Regressió logística": (LogisticRegression(max_iter=10000, random_state=42), True),
    "SVM": (SVC(random_state=42), True),
}

files = []
for nom, (model, escala) in models.items():
    Xtr = ...  # tria X_entrena_esc o X_entrena segons 'escala'
    Xte = ...
    # entrena, prediu, i calcula precisio / sensibilitat / precisio_maligne

    files.append((nom, ..., ..., ...))

taula = pd.DataFrame(
    files, columns=["model", "precisio", "sensibilitat_maligne", "precisio_maligne"]
)
print(taula.sort_values("precisio", ascending=False))
print()
print(taula.sort_values("sensibilitat_maligne", ascending=False))

**Com saps que ho has fet bé:** per precisió, l'ordre és regressió
logística (98,2 %), SVM (97,7 %), bosc aleatori (97,1 %), k-NN (95,9 %) i
arbre de decisió (94,2 %), **l'últim**. Per sensibilitat sobre la classe
maligne, l'ordre canvia: regressió logística (98,4 %), SVM (96,8 %),
**arbre de decisió (95,2 %) puja al tercer lloc**, i k-NN i bosc aleatori
empaten últims (93,7 %). El model que semblava pitjor per precisió és,
de fet, el tercer millor detectant tumors malignes.

### Exercici 6 — Moure el llindar de decisió

Fins ara, `predict()` decideix "maligne" o "benigne" triant la classe amb
més probabilitat, és a dir, tallant a 0,5. Però el 0,5 no és una llei
física: el pots moure.

Entrena una altra `LogisticRegression(max_iter=10000, random_state=42)`,
aquesta vegada **sense escalar** (per tenir un model amb marge de millora
i que l'efecte es vegi clar). Calcula la seva matriu de confusió amb el
llindar per defecte (`.predict()`).

Després, fes servir `predict_proba(X_examen)`: la primera columna és la
probabilitat que el model dona a "maligne" (comprova-ho amb
`model.classes_`, ha de ser `[0, 1]`). En lloc de predir amb `.predict()`,
construeix les teves prediccions dient "maligne" quan aquesta probabilitat
sigui **≥ 0,3** (enlloc de ≥ 0,5) i "benigne" en cas contrari. Calcula la
matriu de confusió d'aquestes noves prediccions.

**Compara les dues matrius: què passa amb els falsos negatius i amb els
falsos positius?**

In [ ]:
# Exercici 6
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix

model_llindar = LogisticRegression(max_iter=10000, random_state=42)
# entrena'l sobre X_entrena / y_entrena, SENSE escalar

prediccions_05 = ...
matriu_05 = confusion_matrix(y_examen, prediccions_05, labels=[0, 1])
print("Llindar per defecte (0,5):")
print(matriu_05)

probabilitats = model_llindar.predict_proba(X_examen)
llindar = 0.3

# prediu 0 (maligne) quan probabilitats[:, 0] >= llindar, si no 1 (benigne)
prediccions_03 = ...

matriu_03 = confusion_matrix(y_examen, prediccions_03, labels=[0, 1])
print("Llindar 0,3:")
print(matriu_03)

**Com saps que ho has fet bé:** amb el llindar per defecte, la matriu és
`[[61, 2], [2, 106]]`: **2 falsos negatius** i 2 falsos positius. Amb el
llindar a 0,3, la matriu passa a `[[62, 1], [4, 104]]`: **1 fals negatiu**
(n'has recuperat un) i **4 falsos positius** (el doble que abans). Baixar
el llindar fa el model més "desconfiat": detecta més malignes reals, però
a canvi genera més falses alarmes sobre benignes. No és gratuït: és un
intercanvi, i **aquest és el moment clau del quadern**.

### Exercici 7 — La decisió final

Tria, d'entre tot el que has provat, un model i un llindar de decisió.
Justifica'l per escrit en tres o quatre línies: **quin error prefereixes
cometre, més falsos positius o més falsos negatius, i per què**, en el
context d'un diagnòstic mèdic. No hi ha una única resposta correcta — es
jutja per l'argument, no pel model triat.

_Escriu aquí la teva decisió (tres o quatre línies)._

**Com saps que ho has fet bé:** no hi ha un número a comprovar aquí. La
resposta és bona si fa servir els números reals dels exercicis anteriors
(falsos negatius i falsos positius de com a mínim dos llindars) i explica
el motiu de la tria, no només el resultat.